### Pydantic

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-20b")

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year movie was released")
    director: str = Field(description="Director of the movie")
    rating: float = Field(description="Movie's rating out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DB52435A30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DB526B4680>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': "Movie's rating out of 10", 

In [7]:
model_with_structure.invoke("Provide details about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [8]:
model.invoke("Provide details about movie Inception")

AIMessage(content='**Inception (2010)**  \n*Directed by*: Christopher\u202fNolan  \n*Produced by*: Emma\u202fThomas, Christopher\u202fNolan  \n*Screenplay*: Christopher\u202fNolan  \n*Music*: Hans\u202fZimmer (original score), additional music by Ludwig\u202fGöransson (end credits)  \n*Production companies*: Syncopy, Warner\u202fBros. Pictures, Legendary Pictures  \n*Distributed by*: Warner\u202fBros. Pictures  \n*Release dates*:  \n- 14\u202fJuly\u202f2010 (Los\u202fAngeles, CA – premiere)  \n- 16\u202fJuly\u202f2010 (United\u202fStates)  \n- 29\u202fJuly\u202f2010 (United\u202fKingdom)  \n\n---\n\n### 1. Premise & Genre\n- **Genre**: Science‑fiction action‑thriller, psychological thriller, heist film  \n- **Premise**: A skilled thief, Dom\u202fCobb, who can enter people’s dreams and steal secrets, is offered a chance to erase his criminal record by planting an idea in someone’s mind—a process known as *inception*. To pull it off, he assembles a team and dives deep into layers of shar

In [9]:
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year movie was released")
    director: str = Field(..., description="Director of the movie")
    rating: float = Field(..., description="Movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

In [10]:
model_with_structure.invoke("Provide details about movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function with director, rating, title, year. Provide details about movie Inception. So we need to call the function with those arguments.', 'tool_calls': [{'id': 'fc_db3a9c7b-72a1-45d0-8f4a-97e38117d2e9', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 155, 'total_tokens': 228, 'completion_time': 0.07937631, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.01049461, 'prompt_tokens_details': None, 'queue_time': 0.049669594, 'total_time': 0.08987092}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dbe61-2cfd-7c71-9d88-97ec81ed80a4-0', tool_calls=[{'name': '

In [11]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    rating: float | None = Field(..., description="Budget in millions USD")

In [12]:
model_with_structure = model.with_structured_output(MovieDetails)

In [13]:
model_with_structure.invoke("Provide details about movie Inception")

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')], genres=['Action', 'Sci-Fi', 'Thriller'], rating=8.8)

### TypeDict

In [14]:
from typing import TypedDict, Annotated

In [15]:
class MovieDict(TypedDict):
    title: Annotated[str, ..., "Title of the movie"]
    year: Annotated[int, ..., "Release year of the movie"]
    director: Annotated[str, ..., "Director of the movie"]
    rating: Annotated[float, ..., "Rating of the movie out of 10"]

In [16]:
model_with_typedict = model.with_structured_output(MovieDict)

In [18]:
model_with_typedict.invoke("Provide details about movie Inception")

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [19]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    rating: float | None = Field(..., description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

model_with_structure.invoke("Provide details about movie Avengers")

{'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'rating': 8,
 'title': 'The Avengers',
 'year': 2012}

In [20]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Classes

In [21]:
from langchain.agents import create_agent

In [22]:
class ContactInfo(BaseModel):
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email Address of the person")
    phone: str = Field(description="Phone number of the person")

In [23]:

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

In [24]:
result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract the contact information: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract the contact information: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2541d844-8cec-4fa8-8624-f1dacff49fa9'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to produce a JSON object that matches the schema ContactInfo. The schema: type object, properties: email (string), name (string), phone (string). Required: name, email, phone. Title: ContactInfo. So output a JSON object with those keys: name, email, phone. Use compact JSON formatting: no whitespace. Ensure valid JSON. Output only the JSON object. Let\'s produce: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 227, 'total_tokens': 366, 'completion_time': 0.149749445, 'completion_tokens_details': {'reasoning_tokens': 107}, 'prompt_time': 0.12351

In [25]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [26]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

In [27]:
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

In [28]:
result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract the contact information: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')